In [1]:
# ============================================================================
# PART 1: INSTALLATIONS AND IMPORTS
# ============================================================================

# Install required libraries
!pip install gradio pandas plotly scikit-learn pyspark findspark -q

import gradio as gr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# PySpark imports
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Medical Insurance Fraud Detection") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Global variables
df_pandas = None
df_spark = None
model_sklearn = None
disease_encoder = None
hospital_encoder = None
available_diseases = []
available_hospitals = []

print("✅ Part 1 Complete: All libraries imported and PySpark initialized!")
print("=" * 80)
print("Now run PART 2...")

# ============================================================================
# PART 2: CORE FUNCTIONS (FIXED FORMATTING VERSION)
# ============================================================================

def load_dataset(file):
    global df_pandas, df_spark, available_diseases, available_hospitals

    if file is None:
        return "Please upload a CSV file", None, None

    df_pandas = pd.read_csv(file.name if hasattr(file, "name") else file)

    required_cols = ['Claim_ID', 'Patient_Name', 'Age', 'Disease', 'Claim_Amount', 'PotentialFraud', 'Hospital']
    if not all(col in df_pandas.columns for col in required_cols):
        return "Error: Missing required columns", None, None

    available_diseases = sorted(df_pandas['Disease'].astype(str).unique().tolist())
    available_hospitals = sorted(df_pandas['Hospital'].astype(str).unique().tolist())

    df_spark = spark.createDataFrame(df_pandas)

    total_records = df_spark.count()
    fraud_count = df_spark.filter(col("PotentialFraud") == 1).count()
    non_fraud_count = total_records - fraud_count

    stats = df_spark.select(
        spark_sum("Claim_Amount").alias("total_claims"),
        avg("Claim_Amount").alias("avg_claims")
    ).collect()[0]

    summary = f"""
✅ Dataset Loaded Successfully! (Using PySpark for processing)

📊 Dataset Summary:
- Total Records: {total_records:,}
- Total Columns: {len(df_pandas.columns)}
- Fraud Cases: {fraud_count:,}
- Non-Fraud Cases: {non_fraud_count:,}
- Fraud Rate: {fraud_count/total_records*100:.2f}%
- Total Claim Amount: ${stats['total_claims']:,.2f}
- Average Claim Amount: ${stats['avg_claims']:,.2f}
- Unique Diseases: {len(available_diseases)}
- Unique Hospitals: {len(available_hospitals)}

🔥 PySpark Engine: ACTIVE
"""

    preview_html = df_pandas.head(10).to_html(index=False, classes='dataframe')

    return summary, preview_html, "Dataset loaded into PySpark successfully!"


def search_claims(search_query, search_type):
    global df_spark

    if df_spark is None:
        return "⚠️ Please upload a dataset first!", None

    search_query = search_query.strip()
    if not search_query:
        return "Please enter a search query", None

    if search_type == "Claim ID":
        results = df_spark.filter(col("Claim_ID").contains(search_query))
    elif search_type == "Patient Name":
        results = df_spark.filter(col("Patient_Name").contains(search_query))
    elif search_type == "Hospital":
        results = df_spark.filter(col("Hospital").contains(search_query))
    elif search_type == "Disease":
        results = df_spark.filter(col("Disease").contains(search_query))
    else:
        return "Invalid search type", None

    results_count = results.count()
    if results_count == 0:
        return f"No results found for '{search_query}' in {search_type}", None

    results_pd = results.toPandas()
    fraud_in_results = results_pd['PotentialFraud'].sum()
    total_amount = results_pd['Claim_Amount'].sum()
    avg_amount = results_pd['Claim_Amount'].mean()

    summary = f"""
🔍 Search Results for: '{search_query}' (Type: {search_type})

📊 Results Summary:
- Total Matches: {results_count}
- Fraud Cases: {fraud_in_results}
- Non-Fraud Cases: {results_count - fraud_in_results}
- Total Claim Amount: ${total_amount:,.2f}
- Average Claim Amount: ${avg_amount:,.2f}
- Fraud Rate in Results: {fraud_in_results/results_count*100:.2f}%
"""
    results_html = results_pd.to_html(index=False, classes='dataframe')
    return summary, results_html


def advanced_query(query_text):
    global df_spark
    if df_spark is None:
        return "⚠️ Please upload dataset first!"

    df_spark.createOrReplaceTempView("fraud_data")
    q = query_text.strip()

    try:
        if q.lower() == "top fraud hospitals":
            sql = """
SELECT Hospital,
       COUNT(*) as Total_Claims,
       SUM(CASE WHEN PotentialFraud = 1 THEN 1 ELSE 0 END) as Fraud_Cases,
       ROUND(SUM(CASE WHEN PotentialFraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as Fraud_Rate,
       ROUND(AVG(Claim_Amount), 2) as Avg_Claim_Amount
FROM fraud_data
GROUP BY Hospital
ORDER BY Fraud_Cases DESC
LIMIT 10
"""
        elif q.lower() == "high value claims":
            sql = """
SELECT *
FROM fraud_data
WHERE Claim_Amount > (SELECT AVG(Claim_Amount) * 1.5 FROM fraud_data)
ORDER BY Claim_Amount DESC
LIMIT 20
"""
        elif q.lower() == "fraud by disease":
            sql = """
SELECT Disease,
       COUNT(*) as Total_Cases,
       SUM(CASE WHEN PotentialFraud = 1 THEN 1 ELSE 0 END) as Fraud_Cases,
       ROUND(AVG(Claim_Amount), 2) as Avg_Amount
FROM fraud_data
GROUP BY Disease
ORDER BY Fraud_Cases DESC
"""
        elif q.lower() == "age risk analysis":
            sql = """
SELECT
  CASE
    WHEN Age < 30 THEN 'Under 30'
    WHEN Age BETWEEN 30 AND 45 THEN '30-45'
    WHEN Age BETWEEN 46 AND 60 THEN '46-60'
    ELSE 'Over 60'
  END as Age_Group,
  COUNT(*) as Total_Claims,
  SUM(CASE WHEN PotentialFraud = 1 THEN 1 ELSE 0 END) as Fraud_Cases,
  ROUND(AVG(Claim_Amount), 2) as Avg_Amount
FROM fraud_data
GROUP BY Age_Group
ORDER BY Fraud_Cases DESC
"""
        else:
            sql = q  # custom query

        result = spark.sql(sql)
        result_pd = result.toPandas()
        html = result_pd.to_html(index=False, classes='dataframe')
        return f"✅ Query executed successfully! ({len(result_pd)} rows returned)\n\n{html}"
    except Exception as e:
        return f"❌ Query Error: {str(e)}\n\nTry predefined queries or valid SQL on table 'fraud_data'."


def create_fraud_overview():
    global df_spark
    if df_spark is None:
        return None
    df_pd = df_spark.toPandas()

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Fraud Distribution',
            'Claim Amount by Fraud Status',
            'Top 10 Diseases by Fraud',
            'Age Distribution by Fraud'
        ),
        specs=[[{'type': 'pie'}, {'type': 'box'}],
               [{'type': 'bar'}, {'type': 'histogram'}]]
    )

    fraud_counts = df_pd['PotentialFraud'].value_counts()
    fig.add_trace(
        go.Pie(
            labels=['Non-Fraud', 'Fraud'],
            values=[fraud_counts.get(0, 0), fraud_counts.get(1, 0)],
            marker=dict(colors=['#2ecc71', '#e74c3c']),
            hole=0.3
        ),
        row=1, col=1
    )

    for status in [0, 1]:
        data = df_pd[df_pd['PotentialFraud'] == status]['Claim_Amount']
        fig.add_trace(
            go.Box(
                y=data,
                name=('Fraud' if status == 1 else 'Non-Fraud'),
                marker_color=('#e74c3c' if status == 1 else '#2ecc71')
            ),
            row=1, col=2
        )

    top_diseases = df_pd[df_pd['PotentialFraud'] == 1]['Disease'].value_counts().head(10)
    fig.add_trace(
        go.Bar(
            x=top_diseases.index,
            y=top_diseases.values,
            marker_color='#e74c3c',
            name='Fraud Count by Disease'
        ),
        row=2, col=1
    )

    for status in [0, 1]:
        data = df_pd[df_pd['PotentialFraud'] == status]['Age']
        fig.add_trace(
            go.Histogram(
                x=data,
                name=('Fraud' if status == 1 else 'Non-Fraud'),
                marker_color=('#e74c3c' if status == 1 else '#2ecc71'),
                opacity=0.7,
                nbinsx=20
            ),
            row=2, col=2
        )

    fig.update_layout(
        height=800,
        showlegend=True,
        title_text="📊 Graph 1: Fraud Detection Overview Dashboard"
    )
    return fig


def get_fraud_statistics():
    global df_spark
    if df_spark is None:
        return "⚠️ Please upload dataset first!"

    total = df_spark.count()
    fraud = df_spark.filter(col("PotentialFraud") == 1).count()

    top_hospitals = df_spark.groupBy("Hospital") \
        .agg(
            count("*").alias("Total_Claims"),
            spark_sum("PotentialFraud").alias("Fraud_Cases")
        ) \
        .orderBy(desc("Fraud_Cases")) \
        .limit(5) \
        .toPandas()

    top_diseases = df_spark.filter(col("PotentialFraud") == 1) \
        .groupBy("Disease") \
        .count() \
        .orderBy(desc("count")) \
        .limit(5) \
        .toPandas()

    # Format tables properly
    hospital_table = top_hospitals.to_string(index=False, justify='left')
    disease_table = top_diseases.to_string(index=False, justify='left')

    stats = f"""
📊 COMPREHENSIVE FRAUD STATISTICS
(Powered by Apache Spark PySpark)

🔢 Overall Metrics:
- Total Claims: {total:,}
- Fraud Cases: {fraud:,}
- Fraud Rate: {fraud/total*100:.2f}%

🏥 Top 5 Hospitals by Fraud Count:

{hospital_table}

🦠 Top 5 Diseases in Fraud Cases:

{disease_table}

🔥 Analysis powered by Apache Spark
"""
    return stats


print("✅ Part 2 Complete: All core functions defined with fixed formatting!")
print("=" * 80)
print("Now run PART 3...")

# ============================================================================
# PART 3: ML MODEL TRAINING + Gradio Interface (FIXED VERSION)
# ============================================================================

def train_sklearn_model():
    global df_pandas, model_sklearn, disease_encoder, hospital_encoder

    if df_pandas is None:
        return "⚠️ Please upload dataset first!", None

    df_model = df_pandas.copy()
    disease_encoder = LabelEncoder()
    hospital_encoder = LabelEncoder()
    df_model['Disease_Encoded'] = disease_encoder.fit_transform(df_model['Disease'].astype(str))
    df_model['Hospital_Encoded'] = hospital_encoder.fit_transform(df_model['Hospital'].astype(str))

    X = df_model[['Age', 'Claim_Amount', 'Disease_Encoded', 'Hospital_Encoded']]
    y = df_model['PotentialFraud']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    model_sklearn = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    model_sklearn.fit(X_train, y_train)

    y_pred = model_sklearn.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud'])
    cm = confusion_matrix(y_test, y_pred)

    feature_importance = pd.DataFrame({
        'Feature': ['Age', 'Claim_Amount', 'Disease', 'Hospital'],
        'Importance': model_sklearn.feature_importances_
    }).sort_values('Importance', ascending=False)

    fig = px.imshow(
        cm,
        labels=dict(x="Predicted", y="Actual", color="Count"),
        x=['Non-Fraud', 'Fraud'],
        y=['Non-Fraud', 'Fraud'],
        title='Confusion Matrix - Scikit-learn Model',
        color_continuous_scale='Blues',
        text_auto=True
    )

    result_text = f"""
✅ Scikit-learn Model Trained Successfully!

📊 Model Performance:
- Accuracy: {accuracy:.2%}
- Training Samples: {len(X_train)}
- Testing Samples: {len(X_test)}

📈 Classification Report:
{report}

🎯 Feature Importance:
{feature_importance.to_string(index=False)}

📦 Model can now handle new diseases/hospitals not in training data.
"""
    return result_text, fig


def predict_fraud_sklearn(age, claim_amount, disease_input, hospital_input, custom_disease, custom_hospital):
    global model_sklearn, disease_encoder, hospital_encoder, df_pandas

    if model_sklearn is None:
        return "⚠️ Please train the model first by going to the 'Train ML Model' tab!"

    # Determine final disease and hospital
    disease = (custom_disease.strip() if disease_input == "Other (Custom)" else disease_input)
    hospital = (custom_hospital.strip() if hospital_input == "Other (Custom)" else hospital_input)

    if disease == "" or hospital == "":
        return "⚠️ Please provide disease and hospital values!"

    disease_warning = ""
    hospital_warning = ""

    if disease in disease_encoder.classes_:
        disease_encoded = int(disease_encoder.transform([disease])[0])
    else:
        disease_encoded = len(disease_encoder.classes_) // 2
        disease_warning = f"⚠️ '{disease}' not in training data - using approximate encoding."

    if hospital in hospital_encoder.classes_:
        hospital_encoded = int(hospital_encoder.transform([hospital])[0])
    else:
        hospital_encoded = len(hospital_encoder.classes_) // 2
        hospital_warning = f"⚠️ '{hospital}' not in training data - using approximate encoding."

    input_vec = np.array([[int(age), float(claim_amount), disease_encoded, hospital_encoded]])
    prediction = model_sklearn.predict(input_vec)[0]
    probability = model_sklearn.predict_proba(input_vec)[0]

    avg_claim = df_pandas['Claim_Amount'].mean()
    claim_ratio = float(claim_amount) / avg_claim
    fraud_age_mean = df_pandas[df_pandas['PotentialFraud'] == 1]['Age'].mean() if len(df_pandas[df_pandas['PotentialFraud'] == 1]) > 0 else 0
    age_risk = abs(int(age) - fraud_age_mean) / fraud_age_mean if fraud_age_mean != 0 else 0

    similar = df_pandas[
        (df_pandas['Disease'] == disease) &
        (df_pandas['Hospital'] == hospital) &
        (df_pandas['Age'].between(int(age)-10, int(age)+10))
    ]
    similar_count = len(similar)
    similar_fraud = int(similar['PotentialFraud'].sum()) if similar_count > 0 else 0
    similar_fraud_rate = similar_fraud / similar_count * 100 if similar_count > 0 else 0

    risk_level = "🟢 LOW RISK"
    if probability[1] > 0.7:
        risk_level = "🔴 HIGH RISK"
    elif probability[1] > 0.4:
        risk_level = "🟡 MEDIUM RISK"

    warnings_section = ""
    if disease_warning or hospital_warning:
        warnings_section = f"\n📝 Notes:\n{disease_warning}\n{hospital_warning}\n"

    result = f"""
{'🚨 FRAUD ALERT!' if prediction == 1 else '✅ Legitimate Claim'}

{risk_level}

📊 Prediction Confidence:
- Fraud Probability: {probability[1]:.1%}
- Non-Fraud Probability: {probability[0]:.1%}
- Model Confidence: {'High' if max(probability) > 0.8 else 'Medium' if max(probability) > 0.6 else 'Low'}

📋 Claim Details:
- Patient Age: {int(age)} years
- Claim Amount: ${float(claim_amount):,.2f}
- Disease: {disease}
- Hospital: {hospital}

🔍 Risk Analysis:
- Claim vs Average: {claim_ratio:.2f}× (Average Claim: ${avg_claim:,.2f})
- Claim Amount Impact: {'Very High' if claim_ratio > 2 else 'High' if claim_ratio > 1.5 else 'Normal'}
- Age Risk Factor: {age_risk:.2f}

📈 Historical Context:
- Similar claims found: {similar_count}
- Fraud in similar claims: {similar_fraud}
- Historical fraud rate: {similar_fraud_rate:.1f}%

💡 Recommendation:
{'⚠️ FLAG FOR IMMEDIATE MANUAL REVIEW - High fraud probability!' if prediction == 1 else '✅ APPROVE - Claim appears legitimate. Standard processing.'}
{warnings_section}
"""
    return result


def handle_disease_choice(choice):
    """Show/hide custom disease input based on dropdown selection"""
    return gr.update(visible=(choice == "Other (Custom)"))

def handle_hospital_choice(choice):
    """Show/hide custom hospital input based on dropdown selection"""
    return gr.update(visible=(choice == "Other (Custom)"))

def update_disease_dropdown():
    """Update disease dropdown with loaded data"""
    global available_diseases
    choices = available_diseases + ["Other (Custom)"] if available_diseases else ["Other (Custom)"]
    return gr.update(choices=choices, value=choices[0] if choices else None)

def update_hospital_dropdown():
    """Update hospital dropdown with loaded data"""
    global available_hospitals
    choices = available_hospitals + ["Other (Custom)"] if available_hospitals else ["Other (Custom)"]
    return gr.update(choices=choices, value=choices[0] if choices else None)


# ============================================================================
# GRADIO INTERFACE (FIXED)
# ============================================================================

with gr.Blocks(title="Medical Fraud Detection System", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏥 Medical Insurance Fraud Detection System")
    gr.Markdown("### 🔥 Powered by Apache PySpark + Machine Learning")
    gr.Markdown("Big Data Analytics with Real-time Fraud Detection")

    with gr.Tab("📂 Data Upload"):
        gr.Markdown("## Upload & Load Dataset into PySpark")

        file_input = gr.File(label="Upload CSV Dataset", file_types=[".csv"], type="filepath")
        load_btn = gr.Button("🚀 Load Dataset into PySpark", variant="primary", size="lg")

        with gr.Row():
            output_text = gr.Textbox(label="Dataset Information", lines=12)
            spark_status = gr.Textbox(label="PySpark Status", lines=3)

        data_preview = gr.HTML(label="Data Preview")

        load_btn.click(
            load_dataset,
            inputs=[file_input],
            outputs=[output_text, data_preview, spark_status]
        )

    with gr.Tab("🔍 Search & Query"):
        gr.Markdown("## Search Claims, Patients, and Providers")

        with gr.Row():
            search_query = gr.Textbox(
                label="Search Query",
                placeholder="Enter Claim ID, Patient Name, Hospital, or Disease"
            )
            search_type = gr.Dropdown(
                choices=["Claim ID", "Patient Name", "Hospital", "Disease"],
                label="Search Type",
                value="Patient Name"
            )

        search_btn = gr.Button("🔎 Search", variant="primary", size="lg")
        search_results = gr.Textbox(label="Search Results Summary", lines=8)
        search_table = gr.HTML(label="Matching Records")

        search_btn.click(
            search_claims,
            inputs=[search_query, search_type],
            outputs=[search_results, search_table]
        )

        gr.Markdown("---")
        gr.Markdown("## 💻 Advanced SQL Query Interface")
        gr.Markdown("""
**Quick Queries (type exactly):**
- `top fraud hospitals` - Hospitals with highest fraud rates
- `high value claims` - Claims above 1.5× average
- `fraud by disease` - Fraud statistics by disease type
- `age risk analysis` - Fraud patterns across age groups

**Custom SQL Example:**
```sql
SELECT Hospital, COUNT(*) as claims FROM fraud_data GROUP BY Hospital
```
""")

        query_input = gr.Textbox(
            label="SQL Query",
            placeholder="Enter query or use quick queries above",
            lines=3
        )
        query_btn = gr.Button("▶️ Execute Query", variant="primary", size="lg")
        query_output = gr.HTML(label="Query Results")

        query_btn.click(advanced_query, inputs=[query_input], outputs=[query_output])

    with gr.Tab("📊 Analytics Dashboard"):
        gr.Markdown("## Comprehensive Fraud Analytics Dashboard")

        stats_btn = gr.Button("📈 Generate Statistics Report", variant="primary", size="lg")
        stats_output = gr.Textbox(label="Fraud Statistics", lines=20)
        stats_btn.click(get_fraud_statistics, outputs=[stats_output])

        gr.Markdown("---")
        gr.Markdown("### 📊 Graph 1: Fraud Overview Dashboard")

        overview_btn = gr.Button("Generate Graph 1: Overview", variant="primary")
        overview_plot = gr.Plot(label="Overview Dashboard")
        overview_btn.click(create_fraud_overview, outputs=[overview_plot])

    with gr.Tab("🤖 Train ML Model"):
        gr.Markdown("## Train Scikit-learn Random Forest Model")
        gr.Markdown("*This model intelligently handles diseases/hospitals not in the training data*")

        train_btn = gr.Button("🔥 Train Model", variant="primary", size="lg")
        model_output = gr.Textbox(label="Training Results", lines=18)
        cm_plot = gr.Plot(label="Confusion Matrix")

        train_btn.click(train_sklearn_model, outputs=[model_output, cm_plot])

    with gr.Tab("🎯 Predict Fraud"):
        gr.Markdown("## Fraud Prediction Interface")
        gr.Markdown("*Enter claim details for real-time fraud risk assessment*")
        gr.Markdown("⚠️ **Important:** Train the model first in the 'Train ML Model' tab")

        with gr.Row():
            age_input = gr.Number(label="Patient Age", value=45, minimum=0, maximum=120)
            claim_input = gr.Number(label="Claim Amount ($)", value=10000, minimum=0)

        gr.Markdown("### Disease Selection")
        gr.Markdown("*Select from dataset or choose 'Other (Custom)' to enter your own*")

        disease_input = gr.Dropdown(
            label="Select Disease",
            choices=["Surgery", "Flu", "Chest Pain", "Back Pain", "Fracture",
                    "Cancer", "Heart Disease", "Diabetes", "Headache", "Allergy",
                    "Migraine", "Pneumonia", "Asthma", "Appendicitis", "Other (Custom)"],
            value="Surgery"
        )

        custom_disease_input = gr.Textbox(
            label="Custom Disease Name",
            placeholder="e.g., Kidney Failure, Liver Disease, etc.",
            visible=False
        )

        gr.Markdown("### Hospital Selection")
        gr.Markdown("*Select from dataset or choose 'Other (Custom)' to enter your own*")

        hospital_input = gr.Dropdown(
            label="Select Hospital",
            choices=["Hospital A", "Hospital B", "Hospital C", "Hospital D",
                    "Clinic A", "Clinic B", "Clinic C", "Clinic D", "Other (Custom)"],
            value="Hospital A"
        )

        custom_hospital_input = gr.Textbox(
            label="Custom Hospital Name",
            placeholder="e.g., City General Hospital, Regional Medical Center, etc.",
            visible=False
        )

        # Connect dropdown changes to show/hide custom inputs
        disease_input.change(
            handle_disease_choice,
            inputs=[disease_input],
            outputs=[custom_disease_input]
        )

        hospital_input.change(
            handle_hospital_choice,
            inputs=[hospital_input],
            outputs=[custom_hospital_input]
        )

        predict_btn = gr.Button("🎯 Predict Fraud Risk", variant="primary", size="lg")
        prediction_output = gr.Textbox(label="Prediction Result", lines=25)

        predict_btn.click(
            predict_fraud_sklearn,
            inputs=[age_input, claim_input, disease_input, hospital_input,
                   custom_disease_input, custom_hospital_input],
            outputs=[prediction_output]
        )

        gr.Markdown("---")
        gr.Markdown("""
### 🧪 Test Examples:
Try these to see how claim amount affects predictions:

**Low Risk Test:**
- Age: 35, Amount: $2,000, Disease: Headache → Should show LOW RISK ✅

**High Risk Test:**
- Age: 35, Amount: $200,000, Disease: Headache → Should show HIGH RISK 🚨

**Custom Input Test:**
- Select "Other (Custom)" for disease → Type "Kidney Failure"
- Select "Other (Custom)" for hospital → Type "City Hospital"
""")

    gr.Markdown("---")
    gr.Markdown("""
    ## 💡 Quick Start Guide:
    1. **Upload Dataset** - Load CSV (must have: Claim_ID, Patient_Name, Age, Disease, Claim_Amount, PotentialFraud, Hospital)
    2. **Search & Query** - Find claims, patients, or run SQL queries
    3. **Analytics** - View fraud statistics and interactive graphs
    4. **Train Model** - Build ML model (handles new diseases/hospitals)
    5. **Predict** - Get real-time fraud predictions with risk analysis

    ## 🎯 Key Features:
    - ✅ **Interactive Graphs** - Visual fraud analytics
    - ✅ **Smart Predictions** - Claim amount affects fraud probability
    - ✅ **Custom Inputs** - Enter diseases/hospitals not in dataset
    - ✅ **Risk Analysis** - Detailed risk factors and recommendations
    - ✅ **PySpark Backend** - Big data processing
    - ✅ **SQL Queries** - Custom analytics queries

    🔥 **Tech Stack:** Apache Spark, PySpark, Scikit-learn, Gradio, Plotly
    """)

print("=" * 80)
print("✅ Part 3 Complete: ML Model and Gradio Interface Ready!")
print("=" * 80)
print("🚀 Launching application...")
print("=" * 80)

demo.launch(debug=False, share=True)

✅ Part 1 Complete: All libraries imported and PySpark initialized!
Now run PART 2...
✅ Part 2 Complete: All core functions defined with fixed formatting!
Now run PART 3...
✅ Part 3 Complete: ML Model and Gradio Interface Ready!
🚀 Launching application...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a9d45655a78cc86bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
